# Final Kaggle Submission


## 1. Imports and path setup


In [ ]:
from pathlib import Path
import json
import random
import time
import warnings
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"XGBoost available: {XGBOOST_AVAILABLE}")
print(f"RANDOM_STATE: {RANDOM_STATE}")


In [ ]:
def list_kaggle_input_tree(root: Path, max_depth: int = 4) -> None:
    print(f"Contents of {root}:")
    if not root.exists():
        print("  (path does not exist)")
        return
    for path in sorted(root.rglob("*")):
        try:
            relative = path.relative_to(root)
        except ValueError:
            continue
        if len(relative.parts) > max_depth:
            continue
        marker = "/" if path.is_dir() else ""
        print(f"  {relative.as_posix()}{marker}")


def count_horizontal_csvs(directory: Path) -> int:
    if not directory.is_dir():
        return 0
    return len(list(directory.rglob("*__horizontal_well.csv")))


def is_well_data_dir(directory: Path) -> bool:
    return count_horizontal_csvs(directory) > 0


def try_pair_from_root(data_root: Path):
    train_dir = data_root / "train"
    test_dir = data_root / "test"
    if train_dir.is_dir() and test_dir.is_dir():
        return data_root, train_dir, test_dir
    return None


def find_train_test_dirs(kaggle_input: Path):
    competition_slug = "rogii-wellbore-geology-prediction"
    explicit_roots = [
        kaggle_input / competition_slug,
        kaggle_input / "competitions" / competition_slug,
    ]
    if kaggle_input.is_dir():
        for child in sorted(kaggle_input.iterdir()):
            if child.is_dir():
                explicit_roots.append(child)
                explicit_roots.append(child / competition_slug)

    seen = set()
    for root in explicit_roots:
        if not root.exists():
            continue
        key = str(root.resolve())
        if key in seen:
            continue
        seen.add(key)
        pair = try_pair_from_root(root)
        if pair is None:
            continue
        data_root, train_dir, test_dir = pair
        if is_well_data_dir(train_dir) and is_well_data_dir(test_dir):
            return data_root, train_dir, test_dir

    candidates = []
    for train_dir in sorted(kaggle_input.rglob("train")):
        if not train_dir.is_dir():
            continue
        test_dir = train_dir.parent / "test"
        if not test_dir.is_dir():
            continue
        if is_well_data_dir(train_dir) and is_well_data_dir(test_dir):
            candidates.append((train_dir.parent, train_dir, test_dir))

    if candidates:
        candidates.sort(key=lambda item: (len(item[0].parts), str(item[0])))
        return candidates[0]

    list_kaggle_input_tree(kaggle_input)
    raise FileNotFoundError(
        "Could not locate train/ and test/ under /kaggle/input. "
        "Attach the competition dataset and re-run."
    )


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for parent in cwd.parents:
        candidates.append(parent)
        if len(candidates) > 6:
            break
    for candidate in candidates:
        if (candidate / "data" / "raw" / "train").is_dir():
            return candidate
        if (candidate / "data" / "train").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root containing a data/ folder with train wells."
    )


def find_sample_submission(search_roots: list) -> Path:
    candidates = []
    for root in search_roots:
        if root is None or not Path(root).exists():
            continue
        root = Path(root)
        candidates.extend(root.rglob("sample_submission.csv"))
        candidates.extend(root.rglob("*submission*.csv"))
    exact = [p for p in candidates if p.name.lower() == "sample_submission.csv"]
    pool = exact if exact else candidates
    if not pool:
        raise FileNotFoundError("sample_submission.csv not found.")
    return sorted(set(pool), key=lambda p: (len(p.parts), str(p)))[0]


KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.exists()

if IS_KAGGLE:
    print("=== Kaggle input tree ===")
    list_kaggle_input_tree(KAGGLE_INPUT)
    DATA_ROOT, TRAIN_DIR, TEST_DIR = find_train_test_dirs(KAGGLE_INPUT)
    PROJECT_ROOT = Path("/kaggle/working")
    WORKING_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = resolve_project_root()
    if (PROJECT_ROOT / "data" / "raw" / "train").is_dir():
        DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    else:
        DATA_ROOT = PROJECT_ROOT / "data"
    TRAIN_DIR = DATA_ROOT / "train"
    TEST_DIR = DATA_ROOT / "test"
    WORKING_ROOT = PROJECT_ROOT

RESULTS_DIR = WORKING_ROOT / "results" / "final_submission"
TABLES_DIR = RESULTS_DIR / "tables"
METADATA_DIR = RESULTS_DIR / "metadata"
SUBMISSIONS_DIR = WORKING_ROOT / "submissions" / "final_submission"

for directory in [RESULTS_DIR, TABLES_DIR, METADATA_DIR, SUBMISSIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_SUBMISSION_PATH = find_sample_submission(
    [TEST_DIR, DATA_ROOT, DATA_ROOT.parent if DATA_ROOT else None, WORKING_ROOT]
)

# Fast profile: one mask/well, capped rows, XGBoost only
MAX_HIDDEN_ROWS_PER_MASK = 400
MODEL_FIT_MAX_ROWS = 150_000
MAX_MASKS_PER_WELL = 1
MIN_KNOWN_ROWS = 50
MIN_HIDDEN_ROWS = 20
XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.05
XGB_EARLY_STOPPING_ROUNDS = 30

print("IS_KAGGLE:", IS_KAGGLE)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print(
    "Speed profile:",
    {
        "MAX_HIDDEN_ROWS_PER_MASK": MAX_HIDDEN_ROWS_PER_MASK,
        "MODEL_FIT_MAX_ROWS": MODEL_FIT_MAX_ROWS,
        "MAX_MASKS_PER_WELL": MAX_MASKS_PER_WELL,
        "XGB_N_ESTIMATORS": XGB_N_ESTIMATORS,
    },
)


## 2. Discover wells and inspect the test missing pattern


In [ ]:
def extract_well_id(path: Path, suffix: str) -> str:
    name = path.name
    if not name.endswith(suffix):
        raise ValueError(f"Unexpected filename {name} for suffix {suffix}")
    return name[: -len(suffix)]


def discover_well_files(directory: Path):
    horizontal_map = {}
    typewell_map = {}
    for path in sorted(directory.rglob("*__horizontal_well.csv")):
        well_id = extract_well_id(path, "__horizontal_well.csv")
        if well_id in horizontal_map:
            raise ValueError(f"Duplicate horizontal well_id {well_id}")
        horizontal_map[well_id] = path
    for path in sorted(directory.rglob("*__typewell.csv")):
        well_id = extract_well_id(path, "__typewell.csv")
        if well_id in typewell_map:
            raise ValueError(f"Duplicate typewell well_id {well_id}")
        typewell_map[well_id] = path
    return horizontal_map, typewell_map


def build_well_registry(horizontal_map, typewell_map, require_pairs: bool = True) -> pd.DataFrame:
    all_ids = sorted(set(horizontal_map) | set(typewell_map))
    rows = []
    for well_id in all_ids:
        h_path = horizontal_map.get(well_id)
        t_path = typewell_map.get(well_id)
        rows.append(
            {
                "well_id": well_id,
                "horizontal_path": str(h_path) if h_path else None,
                "typewell_path": str(t_path) if t_path else None,
                "horizontal_exists": h_path is not None,
                "typewell_exists": t_path is not None,
            }
        )
    registry = pd.DataFrame(rows)
    if require_pairs:
        incomplete = registry[~(registry["horizontal_exists"] & registry["typewell_exists"])]
        if len(incomplete):
            raise ValueError(
                "Incomplete well pairs:\n"
                + incomplete[["well_id", "horizontal_exists", "typewell_exists"]].to_string(index=False)
            )
    return registry


train_horizontal_map, train_typewell_map = discover_well_files(TRAIN_DIR)
test_horizontal_map, test_typewell_map = discover_well_files(TEST_DIR)
train_registry = build_well_registry(train_horizontal_map, train_typewell_map, require_pairs=True)
test_registry = build_well_registry(test_horizontal_map, test_typewell_map, require_pairs=True)

sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert list(sample_submission_df.columns) == ["id", "tvt"], sample_submission_df.columns.tolist()

print(f"Train wells: {len(train_registry)}")
print(f"Test wells: {len(test_registry)}")
print(f"Sample submission rows: {len(sample_submission_df)}")
display(test_registry)


In [ ]:
def analyze_known_missing(horizontal_df: pd.DataFrame, well_id: str) -> dict:
    n = len(horizontal_df)
    missing = horizontal_df["TVT_input"].isna().to_numpy()
    n_missing = int(missing.sum())
    n_known = n - n_missing
    first_missing = int(np.argmax(missing)) if n_missing else n
    trailing = bool(n_missing and missing[first_missing:].all() and (~missing[:first_missing]).all())
    known_tvt = horizontal_df.loc[~missing, "TVT_input"] if n_known else pd.Series(dtype=float)
    return {
        "well_id": well_id,
        "total_rows": n,
        "known_rows": n_known,
        "missing_rows": n_missing,
        "missing_fraction": n_missing / n if n else np.nan,
        "trailing_missing_interval": trailing,
        "last_known_tvt": float(known_tvt.iloc[-1]) if n_known else np.nan,
        "last_known_md": float(horizontal_df.loc[~missing, "MD"].iloc[-1]) if n_known else np.nan,
    }


test_horizontal_frames = {}
known_missing_rows = []
for _, row in test_registry.iterrows():
    well_id = row["well_id"]
    h_df = pd.read_csv(row["horizontal_path"])
    test_horizontal_frames[well_id] = h_df
    stats = analyze_known_missing(h_df, well_id)
    known_missing_rows.append(stats)
    if not stats["trailing_missing_interval"]:
        print(f"WARNING: {well_id} missing rows are not a strict trailing interval")

known_missing_df = pd.DataFrame(known_missing_rows)
TEST_MISSING_FRACTIONS = known_missing_df["missing_fraction"].dropna().tolist()
display(known_missing_df)
print("Test missing fractions:", [round(x, 3) for x in TEST_MISSING_FRACTIONS])
assert known_missing_df["trailing_missing_interval"].all()


## 3. Feature engineering

Prediction-time-safe horizontal features plus last-known TVT / slope projections. No typewell nearest-neighbor features.


In [ ]:
RAW_HORIZONTAL_FEATURES = ["MD", "GR", "X", "Y", "Z"]
FORMATION_COLUMNS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
ROLLING_WINDOWS = [5, 11]
SLOPE_WINDOWS = [10, 25, 50]
PRIMARY_SLOPE_WINDOW = 50


def engineer_horizontal_features(horizontal_df: pd.DataFrame, well_id: str) -> pd.DataFrame:
    missing = [c for c in RAW_HORIZONTAL_FEATURES if c not in horizontal_df.columns]
    if missing:
        raise ValueError(f"Well {well_id} missing required columns: {missing}")

    df = horizontal_df.copy()
    df["original_row_index"] = df.index.astype(int)
    df["well_id"] = well_id
    df = df.sort_values("MD").reset_index(drop=True)

    md0 = df["MD"].iloc[0]
    df["MD_relative"] = df["MD"] - md0
    md_range = float(df["MD"].iloc[-1] - md0)
    df["MD_normalized"] = df["MD_relative"] / md_range if md_range > 0 else 0.0

    df["X_relative"] = df["X"] - df["X"].iloc[0]
    df["Y_relative"] = df["Y"] - df["Y"].iloc[0]
    df["Z_relative"] = df["Z"] - df["Z"].iloc[0]

    for col in ["MD", "X", "Y", "Z", "GR"]:
        df[f"{col}_diff"] = df[col].diff()

    df["step_distance_3d"] = np.sqrt(
        df["X_diff"].fillna(0.0) ** 2
        + df["Y_diff"].fillna(0.0) ** 2
        + df["Z_diff"].fillna(0.0) ** 2
    )
    df["cumulative_trajectory_distance"] = df["step_distance_3d"].cumsum()
    df["horizontal_distance"] = np.sqrt(df["X_relative"] ** 2 + df["Y_relative"] ** 2)
    df["spatial_distance"] = np.sqrt(
        df["X_relative"] ** 2 + df["Y_relative"] ** 2 + df["Z_relative"] ** 2
    )

    for window in ROLLING_WINDOWS:
        roll = df["GR"].rolling(window=window, min_periods=1)
        df[f"GR_roll_mean_{window}"] = roll.mean()
        df[f"GR_roll_std_{window}"] = roll.std()
        df[f"GR_dev_from_roll_mean_{window}"] = df["GR"] - df[f"GR_roll_mean_{window}"]

    with np.errstate(divide="ignore", invalid="ignore"):
        df["dz_dmd"] = df["Z_diff"] / df["MD_diff"].replace(0, np.nan)

    return df


def _safe_polyfit_slope(md_values: np.ndarray, tvt_values: np.ndarray) -> float:
    if len(md_values) < 2 or np.unique(md_values).size < 2:
        return 0.0
    try:
        slope, _ = np.polyfit(md_values, tvt_values, 1)
        return float(slope) if np.isfinite(slope) else 0.0
    except Exception:
        return 0.0


def compute_slope_from_known(known_md, known_tvt, window: int) -> float:
    if len(known_md) == 0:
        return 0.0
    take = min(window, len(known_md))
    return _safe_polyfit_slope(known_md[-take:], known_tvt[-take:])


def add_last_known_and_slope_features(
    sorted_df: pd.DataFrame,
    known_mask,
    slope_windows=None,
    primary_window: int = PRIMARY_SLOPE_WINDOW,
    tvt_source_column: str = "TVT_input",
) -> pd.DataFrame:
    if slope_windows is None:
        slope_windows = SLOPE_WINDOWS

    df = sorted_df.copy()
    known_mask = np.asarray(known_mask, dtype=bool)
    known_md = df.loc[known_mask, "MD"].to_numpy(dtype=float)
    known_tvt = df.loc[known_mask, tvt_source_column].to_numpy(dtype=float)
    valid = np.isfinite(known_md) & np.isfinite(known_tvt)
    known_md = known_md[valid]
    known_tvt = known_tvt[valid]

    if len(known_tvt) == 0:
        last_known_tvt = np.nan
        last_known_md = np.nan
        first_known_md = np.nan
    else:
        last_known_tvt = float(known_tvt[-1])
        last_known_md = float(known_md[-1])
        first_known_md = float(known_md[0])

    slopes = {w: compute_slope_from_known(known_md, known_tvt, w) for w in slope_windows}
    primary_slope = slopes.get(primary_window, 0.0)
    slope_values = np.array(list(slopes.values()), dtype=float)

    df["last_known_tvt"] = last_known_tvt
    df["last_known_md"] = last_known_md
    df["distance_from_last_known_md"] = df["MD"] - last_known_md
    df["known_tvt_count"] = float(len(known_tvt))
    df["known_fraction"] = float(len(known_tvt) / len(df)) if len(df) else 0.0
    df["md_relative_to_first_known"] = df["MD"] - first_known_md
    df["distance_beyond_known_interval"] = np.maximum(df["MD"] - last_known_md, 0.0)

    for window, slope in slopes.items():
        df[f"tvt_slope_w{window}"] = slope
        df[f"linear_proj_w{window}"] = last_known_tvt + slope * (df["MD"] - last_known_md)

    df["recent_tvt_slope"] = primary_slope
    df["linear_tvt_projection"] = last_known_tvt + primary_slope * (df["MD"] - last_known_md)
    df["slope_mean"] = float(np.mean(slope_values)) if len(slope_values) else 0.0
    df["slope_std"] = float(np.std(slope_values)) if len(slope_values) else 0.0
    df["slope_median"] = float(np.median(slope_values)) if len(slope_values) else 0.0
    if len(slope_windows) >= 2:
        df["slope_long_minus_short"] = slopes[max(slope_windows)] - slopes[min(slope_windows)]
    else:
        df["slope_long_minus_short"] = 0.0
    return df


print("Feature helpers ready.")


## 4. Build a lightweight masked training set

One mask per well, capped hidden rows, no typewell features. This is the main runtime saver versus the heavy submission pipeline.


In [ ]:
def load_csv(path) -> pd.DataFrame:
    return pd.read_csv(path)


def prepare_well_frame(horizontal_df: pd.DataFrame, well_id: str) -> pd.DataFrame:
    raw = horizontal_df.copy()
    raw["original_row_index"] = raw.index.astype(int)
    feat = engineer_horizontal_features(raw, well_id)
    raw_sorted = raw.sort_values("MD").reset_index(drop=True)
    for col in ["TVT_input", "TVT"] + FORMATION_COLUMNS:
        if col in raw_sorted.columns:
            feat[col] = raw_sorted[col].to_numpy()
        elif col not in feat.columns:
            feat[col] = np.nan
    return feat


def find_natural_cut(sorted_df: pd.DataFrame):
    if "TVT_input" not in sorted_df.columns:
        return None
    missing = sorted_df["TVT_input"].isna().to_numpy()
    if not missing.any():
        return None
    first = int(np.argmax(missing))
    if missing[first:].all() and (~missing[:first]).all():
        return first
    return None


def choose_cut(n_rows: int, natural_cut, test_fractions, rng):
    candidates = []

    def try_add(name, cut):
        if cut < MIN_KNOWN_ROWS or (n_rows - cut) < MIN_HIDDEN_ROWS:
            return
        candidates.append((name, cut, (n_rows - cut) / n_rows))

    if natural_cut is not None:
        try_add("natural", natural_cut)

    frac_pool = sorted(
        set(round(float(f), 4) for f in list(test_fractions) + [0.70, 0.75] if np.isfinite(f))
    )
    rng.shuffle(frac_pool)
    for frac in frac_pool:
        cut = int(round(n_rows * (1.0 - frac)))
        cut = max(MIN_KNOWN_ROWS, min(cut, n_rows - MIN_HIDDEN_ROWS))
        try_add(f"frac_{frac:.2f}", cut)
        if len(candidates) >= 4:
            break

    seen = set()
    selected = []
    for item in candidates:
        if item[1] in seen:
            continue
        seen.add(item[1])
        selected.append(item)
        if len(selected) >= MAX_MASKS_PER_WELL:
            break
    return selected[0] if selected else None


def _subsample_hidden(hidden: pd.DataFrame, max_rows: int) -> pd.DataFrame:
    if len(hidden) <= max_rows:
        return hidden
    idx = np.linspace(0, len(hidden) - 1, max_rows).astype(int)
    return hidden.iloc[idx].copy()


def build_masked_rows_for_well(horizontal_df, well_id, test_fractions, rng):
    feat = prepare_well_frame(horizontal_df, well_id)
    usable = feat["TVT"].notna() if "TVT" in feat.columns else pd.Series(False, index=feat.index)
    if usable.sum() < (MIN_KNOWN_ROWS + MIN_HIDDEN_ROWS):
        return pd.DataFrame(), None

    feat = feat.loc[usable].reset_index(drop=True)
    n_rows = len(feat)
    cut_info = choose_cut(n_rows, find_natural_cut(feat), test_fractions, rng)
    if cut_info is None:
        return pd.DataFrame(), None

    mask_name, cut, hidden_frac = cut_info
    known_mask = np.zeros(n_rows, dtype=bool)
    known_mask[:cut] = True

    sim = feat.copy()
    sim["sim_TVT_input"] = np.nan
    sim.loc[known_mask, "sim_TVT_input"] = sim.loc[known_mask, "TVT"].to_numpy()
    slope_frame = add_last_known_and_slope_features(
        sim,
        known_mask,
        slope_windows=SLOPE_WINDOWS,
        primary_window=PRIMARY_SLOPE_WINDOW,
        tvt_source_column="sim_TVT_input",
    )

    hidden = slope_frame.loc[~known_mask].copy()
    ok = np.isfinite(hidden["TVT"].to_numpy(dtype=float)) & np.isfinite(
        hidden["linear_tvt_projection"].to_numpy(dtype=float)
    )
    hidden = _subsample_hidden(hidden.loc[ok], MAX_HIDDEN_ROWS_PER_MASK)
    if hidden.empty:
        return pd.DataFrame(), None

    hidden["well_id"] = well_id
    hidden["mask_id"] = f"{well_id}__{mask_name}"
    hidden["actual_tvt"] = hidden["TVT"].astype(float)
    hidden["residual_target"] = hidden["actual_tvt"] - hidden["linear_tvt_projection"].astype(float)
    hidden["group_id"] = well_id

    summary = {
        "well_id": well_id,
        "mask_name": mask_name,
        "known_rows": int(cut),
        "hidden_rows_used": int(len(hidden)),
        "hidden_fraction": float(hidden_frac),
    }
    return hidden, summary


def build_dataset_for_wells(well_ids, registry, test_fractions, random_state: int):
    rng = np.random.default_rng(random_state)
    frames = []
    summaries = []
    indexed = registry.set_index("well_id")
    t0 = time.time()
    for i, well_id in enumerate(well_ids):
        h_df = load_csv(indexed.loc[well_id, "horizontal_path"])
        rows_df, summary = build_masked_rows_for_well(h_df, well_id, test_fractions, rng)
        if summary is not None:
            summaries.append(summary)
        if not rows_df.empty:
            frames.append(rows_df)
        if (i + 1) % 100 == 0 or (i + 1) == len(well_ids):
            n_rows = sum(len(f) for f in frames)
            print(f"  processed {i + 1}/{len(well_ids)} wells; rows={n_rows:,} ({time.time() - t0:.1f}s)")
        del h_df, rows_df
    if not frames:
        return pd.DataFrame(), pd.DataFrame(summaries)
    return pd.concat(frames, ignore_index=True), pd.DataFrame(summaries)


all_train_well_ids = train_registry["well_id"].tolist()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
_dummy = np.zeros(len(all_train_well_ids))
_groups = np.array(all_train_well_ids)
train_idx, val_idx = next(gss.split(_dummy, groups=_groups))
SPLIT_TRAIN_WELLS = sorted(_groups[train_idx].tolist())
SPLIT_VAL_WELLS = sorted(_groups[val_idx].tolist())

print(f"Train wells: {len(SPLIT_TRAIN_WELLS)} | Val wells: {len(SPLIT_VAL_WELLS)}")
print("Building masked datasets...")
train_masked_df, train_mask_summary_df = build_dataset_for_wells(
    SPLIT_TRAIN_WELLS, train_registry, TEST_MISSING_FRACTIONS, RANDOM_STATE
)
val_masked_df, val_mask_summary_df = build_dataset_for_wells(
    SPLIT_VAL_WELLS, train_registry, TEST_MISSING_FRACTIONS, RANDOM_STATE + 1
)

print(f"Train rows: {len(train_masked_df):,} | Val rows: {len(val_masked_df):,}")
display(pd.concat([train_mask_summary_df, val_mask_summary_df], ignore_index=True).head())


## 5. Validate baselines and train a fast XGBoost residual model


In [ ]:
def calculate_rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def calculate_validation_metrics(y_true, y_pred, well_ids, model_name: str):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    well_ids = np.asarray(well_ids)
    overall = {
        "model": model_name,
        "rmse": calculate_rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "n_rows": int(len(y_true)),
        "n_wells": int(pd.Series(well_ids).nunique()),
    }
    frame = pd.DataFrame({"well_id": well_ids, "y_true": y_true, "y_pred": y_pred})
    records = []
    for well_id, g in frame.groupby("well_id", sort=True):
        records.append(
            {
                "well_id": well_id,
                "rmse": calculate_rmse(g["y_true"], g["y_pred"]),
                "mae": float(mean_absolute_error(g["y_true"], g["y_pred"])),
                "n_rows": len(g),
            }
        )
    per_well = pd.DataFrame(records)
    overall["mean_well_rmse"] = float(per_well["rmse"].mean())
    overall["median_well_rmse"] = float(per_well["rmse"].median())
    overall["max_well_rmse"] = float(per_well["rmse"].max())
    per_well["model"] = model_name
    return overall, per_well


META_COLUMNS = {
    "well_id", "mask_id", "group_id", "original_row_index",
    "actual_tvt", "residual_target", "TVT", "TVT_input", "sim_TVT_input",
}


def infer_feature_columns(df: pd.DataFrame) -> list:
    cols = []
    for col in df.columns:
        if col in META_COLUMNS:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    return sorted(set(cols))


FEATURE_COLUMNS = infer_feature_columns(train_masked_df)
for required in ["MD", "GR", "linear_tvt_projection", "last_known_tvt"]:
    if required in train_masked_df.columns and required not in FEATURE_COLUMNS:
        FEATURE_COLUMNS.append(required)
FEATURE_COLUMNS = sorted(set(FEATURE_COLUMNS))

y_val = val_masked_df["actual_tvt"].to_numpy(dtype=float)
wells_val = val_masked_df["well_id"].to_numpy()

comparison_rows = []
baseline_preds = {
    "last_known_tvt": val_masked_df["last_known_tvt"].to_numpy(dtype=float),
    "linear_projection": val_masked_df["linear_tvt_projection"].to_numpy(dtype=float),
}
for name, preds in baseline_preds.items():
    overall, _ = calculate_validation_metrics(y_val, preds, wells_val, name)
    comparison_rows.append(overall)
    print(f"{name}: RMSE={overall['rmse']:.4f} MAE={overall['mae']:.4f}")

print(f"Features: {len(FEATURE_COLUMNS)}")


In [ ]:
def fit_median_imputer(train_df: pd.DataFrame, columns: list) -> pd.Series:
    return train_df[columns].median(numeric_only=True)


def transform_with_medians(df: pd.DataFrame, columns: list, medians: pd.Series) -> np.ndarray:
    return df[columns].fillna(medians).to_numpy(dtype=np.float32)


train_medians = fit_median_imputer(train_masked_df, FEATURE_COLUMNS)
fit_n = min(int(MODEL_FIT_MAX_ROWS), len(train_masked_df))
fit_df = (
    train_masked_df.sample(n=fit_n, random_state=RANDOM_STATE)
    if fit_n < len(train_masked_df)
    else train_masked_df
)
print(f"Fit rows: {len(fit_df):,} / {len(train_masked_df):,}")

X_fit = transform_with_medians(fit_df, FEATURE_COLUMNS, train_medians)
y_fit = fit_df["residual_target"].to_numpy(dtype=np.float32)
X_val = transform_with_medians(val_masked_df, FEATURE_COLUMNS, train_medians)
y_val_residual = val_masked_df["residual_target"].to_numpy(dtype=np.float32)
val_proj = val_masked_df["linear_tvt_projection"].to_numpy(dtype=float)

xgb_model = None
xgb_val_pred = None

if XGBOOST_AVAILABLE:
    xgb_model = XGBRegressor(
        n_estimators=int(XGB_N_ESTIMATORS),
        max_depth=int(XGB_MAX_DEPTH),
        learning_rate=float(XGB_LEARNING_RATE),
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=1.0,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    t0 = time.time()
    try:
        xgb_model.set_params(early_stopping_rounds=int(XGB_EARLY_STOPPING_ROUNDS))
        xgb_model.fit(X_fit, y_fit, eval_set=[(X_val, y_val_residual)], verbose=False)
    except Exception:
        try:
            xgb_model.set_params(early_stopping_rounds=None)
        except Exception:
            pass
        xgb_model.fit(X_fit, y_fit)
    print(f"XGBoost trained in {time.time() - t0:.1f}s")
    xgb_val_pred = val_proj + xgb_model.predict(X_val)
    overall, _ = calculate_validation_metrics(y_val, xgb_val_pred, wells_val, "xgboost_residual")
    comparison_rows.append(overall)
    print(f"xgboost_residual: RMSE={overall['rmse']:.4f} MAE={overall['mae']:.4f}")
else:
    print("XGBoost unavailable — using domain baselines only.")

# Conservative blend: mostly last-known, a little model signal
if xgb_val_pred is not None:
    last_known = val_masked_df["last_known_tvt"].to_numpy(dtype=float)
    for alpha in [0.85, 0.70]:
        blend = alpha * last_known + (1.0 - alpha) * xgb_val_pred
        name = f"blend_lastknown_{alpha:.2f}_xgb"
        overall, _ = calculate_validation_metrics(y_val, blend, wells_val, name)
        comparison_rows.append(overall)
        print(f"{name}: RMSE={overall['rmse']:.4f}")

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ["rmse", "mean_well_rmse"]
).reset_index(drop=True)
display(comparison_df)
comparison_df.to_csv(TABLES_DIR / "validation_comparison.csv", index=False)

BEST_MODEL = comparison_df.iloc[0]["model"]
BEST_RMSE = float(comparison_df.iloc[0]["rmse"])
print(f"Selected model: {BEST_MODEL} (val RMSE={BEST_RMSE:.4f})")


## 6. Refit on all masked rows and predict the test wells


In [ ]:
# Refit medians / XGBoost on train+val masked rows (still capped)
full_train_df = pd.concat([train_masked_df, val_masked_df], ignore_index=True)
final_medians = fit_median_imputer(full_train_df, FEATURE_COLUMNS)
final_fit_n = min(int(MODEL_FIT_MAX_ROWS), len(full_train_df))
final_fit_df = (
    full_train_df.sample(n=final_fit_n, random_state=RANDOM_STATE)
    if final_fit_n < len(full_train_df)
    else full_train_df
)

final_xgb_model = None
if XGBOOST_AVAILABLE and (
    str(BEST_MODEL).startswith("xgboost") or str(BEST_MODEL).startswith("blend_")
):
    X_full = transform_with_medians(final_fit_df, FEATURE_COLUMNS, final_medians)
    y_full = final_fit_df["residual_target"].to_numpy(dtype=np.float32)
    final_xgb_model = XGBRegressor(
        n_estimators=int(XGB_N_ESTIMATORS),
        max_depth=int(XGB_MAX_DEPTH),
        learning_rate=float(XGB_LEARNING_RATE),
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=1.0,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    t0 = time.time()
    final_xgb_model.fit(X_full, y_full)
    print(f"Final XGBoost refit in {time.time() - t0:.1f}s on {len(final_fit_df):,} rows")
else:
    print("Skipping final XGBoost refit (selected candidate does not need it).")

with open(METADATA_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)


In [ ]:
def construct_test_predictions_for_well(well_id: str) -> pd.DataFrame:
    h_path = test_registry.loc[test_registry["well_id"] == well_id, "horizontal_path"].iloc[0]
    horizontal_df = load_csv(h_path)
    feat = prepare_well_frame(horizontal_df, well_id)
    known_mask = feat["TVT_input"].notna().to_numpy()
    feat = add_last_known_and_slope_features(
        feat,
        known_mask,
        slope_windows=SLOPE_WINDOWS,
        primary_window=PRIMARY_SLOPE_WINDOW,
        tvt_source_column="TVT_input",
    )
    pred_df = feat.loc[~known_mask].copy()
    assert len(pred_df) > 0, f"No missing TVT_input rows for well {well_id}"

    for col in FEATURE_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = np.nan

    out = pd.DataFrame(
        {
            "id": [f"{well_id}_{int(idx)}" for idx in pred_df["original_row_index"].to_numpy()],
            "well_id": well_id,
            "original_row_index": pred_df["original_row_index"].to_numpy(dtype=int),
            "MD": pred_df["MD"].to_numpy(dtype=float),
            "pred_last_known": pred_df["last_known_tvt"].to_numpy(dtype=float),
            "pred_linear": pred_df["linear_tvt_projection"].to_numpy(dtype=float),
        }
    )

    if final_xgb_model is not None:
        X = transform_with_medians(pred_df, FEATURE_COLUMNS, final_medians)
        out["pred_xgboost"] = out["pred_linear"] + final_xgb_model.predict(X)
    return out


test_frames = []
for well_id in test_registry["well_id"].tolist():
    frame = construct_test_predictions_for_well(well_id)
    print(f"{well_id}: {len(frame):,} prediction rows")
    test_frames.append(frame)

test_predictions_df = pd.concat(test_frames, ignore_index=True)
assert test_predictions_df["id"].is_unique
assert len(test_predictions_df) == int(known_missing_df["missing_rows"].sum())
display(test_predictions_df.head())


## 7. Build and validate `submission.csv`


In [ ]:
def resolve_final_predictions(df: pd.DataFrame, model_name: str) -> np.ndarray:
    if model_name == "last_known_tvt":
        return df["pred_last_known"].to_numpy(dtype=float)
    if model_name == "linear_projection":
        return df["pred_linear"].to_numpy(dtype=float)
    if model_name == "xgboost_residual":
        if "pred_xgboost" not in df.columns:
            raise RuntimeError("XGBoost predictions unavailable")
        return df["pred_xgboost"].to_numpy(dtype=float)
    if model_name.startswith("blend_lastknown_"):
        # blend_lastknown_0.85_xgb
        alpha = float(model_name.split("_")[2])
        if "pred_xgboost" not in df.columns:
            raise RuntimeError("XGBoost predictions unavailable for blend")
        return alpha * df["pred_last_known"].to_numpy(dtype=float) + (
            1.0 - alpha
        ) * df["pred_xgboost"].to_numpy(dtype=float)
    raise KeyError(f"Unknown selected model: {model_name}")


pred_values = resolve_final_predictions(test_predictions_df, BEST_MODEL)
pred_map = dict(zip(test_predictions_df["id"].astype(str), pred_values.tolist()))

submission_df = sample_submission_df.copy()
submission_df["id"] = submission_df["id"].astype(str)
missing_ids = [i for i in submission_df["id"] if i not in pred_map]
assert not missing_ids, f"Missing predictions for {len(missing_ids)} sample IDs (e.g. {missing_ids[:3]})"
submission_df["tvt"] = submission_df["id"].map(pred_map)

assert list(submission_df.columns) == ["id", "tvt"]
assert len(submission_df) == len(sample_submission_df)
assert submission_df["id"].is_unique
assert submission_df["tvt"].notna().all()
assert np.isfinite(submission_df["tvt"].to_numpy(dtype=float)).all()
assert submission_df["id"].tolist() == sample_submission_df["id"].astype(str).tolist()

# Confirm only missing TVT_input rows are predicted
independent_ids = []
for well_id, h_df in test_horizontal_frames.items():
    for idx in h_df.index[h_df["TVT_input"].isna()].tolist():
        independent_ids.append(f"{well_id}_{int(idx)}")
assert set(independent_ids) == set(submission_df["id"])

if IS_KAGGLE:
    FINAL_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
else:
    FINAL_SUBMISSION_PATH = SUBMISSIONS_DIR / "submission.csv"

FINAL_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(FINAL_SUBMISSION_PATH, index=False)

# Convenience local copy
local_copy = WORKING_ROOT / "submissions" / "submission.csv"
local_copy.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(local_copy, index=False)

summary = {
    "selected_model": BEST_MODEL,
    "validation_rmse": BEST_RMSE,
    "rows": int(len(submission_df)),
    "prediction_min": float(submission_df["tvt"].min()),
    "prediction_max": float(submission_df["tvt"].max()),
    "prediction_mean": float(submission_df["tvt"].mean()),
    "output_path": str(FINAL_SUBMISSION_PATH),
}
with open(METADATA_DIR / "submission_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Wrote:", FINAL_SUBMISSION_PATH)
print("Also wrote:", local_copy)
print(json.dumps(summary, indent=2))
display(submission_df.head())
display(submission_df.tail())


## 8. Summary

- Built a realistic trailing-mask validation set with **one mask per well** and capped rows
- Compared `last_known_tvt`, linear projection, fast XGBoost residual, and conservative blends
- Selected the best validation candidate and wrote competition-ready `submission.csv`

Expected runtime on Kaggle is typically **under ~20–30 minutes**, dominated by the masked dataset build rather than model training.
